In [ ]:
# STEP 1: Load an embedding model.
# An "embedding model" turns a piece of text into a vector (a list of numbers)
# that captures its meaning. Texts with similar meanings produce similar vectors.
# 'all-MiniLM-L6-v2' is a small, fast model that outputs 384-dimensional vectors.
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
# STEP 2: Load the documents we want to search over.
# load_faq_data() returns a list of FAQ entries; each entry is a dict
# with fields like 'question' and 'answer'. These are our "knowledge base".
from ingest import load_faq_data

documents = load_faq_data()


In [ ]:
# STEP 3: Build the text we want to embed for each document.
# We combine the question and the answer into a single string so the
# resulting vector reflects BOTH what is asked and what is answered.
texts = []

for doc in documents:
    text = doc['question'] + ' ' + doc['answer']
    texts.append(text)


In [ ]:
# tqdm just gives us a nice progress bar while encoding many documents.
from tqdm.auto import tqdm


In [ ]:
# STEP 4: Encode every document into a vector.
# We process texts in small batches (50 at a time) which is faster and
# uses less memory than encoding one-by-one or all-at-once.
# Result: `vectors` is a list with one 384-d vector per document.
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)  # should equal the number of documents


  0%|          | 0/25 [00:00<?, ?it/s]

1208

In [ ]:
# STEP 5: Stack all document vectors into a single NumPy matrix `X`.
# Shape of X = (number_of_documents, 384). Having one big matrix lets us
# compare a query against ALL documents at once using fast matrix math.
import numpy as np
X = np.array(vectors)


In [ ]:
# STEP 6: Take the user's question and turn it into a vector too,
# using the SAME model (important — both sides must live in the same vector space).
query = 'Can I still join the course after the start date?'
v_query = model.encode(query)


In [ ]:
# STEP 7: Score every document against the query in one shot.
# X.dot(v_query) computes the dot product between the query vector and
# each document vector. Because the model produces normalized vectors,
# the dot product is the cosine similarity:
#   higher score  =>  more similar in meaning.
scores = X.dot(v_query)


In [ ]:
# STEP 8: Find the BEST matching document — the one with the highest score.
# np.argmax returns the position (index) of the largest value in `scores`.
idx = np.argmax(scores)
idx, scores[idx]  # (which document, how similar it is)


(np.int64(553), np.float32(0.762941))

In [ ]:
# STEP 9: Look up the original document at that index.
# This is the FAQ entry our vector search judged most relevant to the query.
documents[idx]


{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}